# Extreme Heat Event Analysis Template

Analyzes an extreme heat event using Cal-Adapt's `climakitae` library. Location, event
dates, thresholds, and warming levels are all read from `event_config.yaml` — nothing
below is hardcoded to a specific place or event.

## How to use this template

1. Copy `extreme_heat_event_template.ipynb`, `run_event.sh`, and `event_config.example.yaml`
   into a new folder (e.g. `blog/2026-XX-XX-newcity-extreme-heat/`), renaming the config
   copy to `event_config.yaml`.
2. Edit `event_config.yaml`: station/location, the heat wave date window(s), thresholds,
   warming levels, and (if you're building the Part 4 CRAI maps) the CRAI file paths. Any
   zip/CSV assets the config points to (a zip-code shapefile, CRAI CSVs) go in that same
   folder.
3. Run it: `bash run_event.sh event_config.yaml extreme_heat_event_template.ipynb` — runs
   the notebook end-to-end under the `climakitae` conda env via `jupyter execute --inplace`;
   figures land in `figures/static/` and `figures/html/`.

**Heat event definition** (see `thresholds` in `event_config.yaml`): a day qualifies when
daily max temperature exceeds `tmax_f` **and** daily min temperature is at or above
`tmin_f`.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
import climakitae as ck
from dask.diagnostics import ProgressBar

# All figure exports below write into these two folders (relative to the
# kernel's cwd) — create them up front so fig.write_html/write_image don't
# fail with FileNotFoundError on a fresh checkout or a remote kernel whose
# cwd isn't this notebook's own directory.
os.makedirs("figures/html", exist_ok=True)
os.makedirs("figures/static", exist_ok=True)

plotly_config = dict(
    responsive=True,
    displaylogo=False,
    scrollZoom=False,
    modeBarButtonsToRemove=[
        "zoom2d", "pan2d", "select2d", "lasso2d",
        "zoomIn2d", "zoomOut2d", "autoScale2d", "resetScale2d",
        "zoomInGeo", "zoomOutGeo", "resetGeo", "hoverClosestGeo",
    ],
)

In [ ]:
import yaml

# Path to the event config — override by setting EXTREME_HEAT_CONFIG before
# running (run_event.sh does this), or just drop an event_config.yaml next to
# this notebook for interactive use.
CONFIG_PATH = os.environ.get("EXTREME_HEAT_CONFIG", "event_config.yaml")
with open(CONFIG_PATH) as f:
    CFG = yaml.safe_load(f)


def _require(cfg, *path):
    node = cfg
    for key in path:
        if not isinstance(node, dict) or key not in node:
            raise KeyError(
                f"Missing required config key: {'.'.join(path)} "
                f"(loaded from {CONFIG_PATH})"
            )
        node = node[key]
    return node


EVENT_NAME = _require(CFG, "event_name")

NETWORK_ID = _require(CFG, "location", "network_id")
STATION_ID = _require(CFG, "location", "station_id")
STATION_LABEL = _require(CFG, "location", "station_label")
PLACE_NAME = _require(CFG, "location", "place_name")
CLIP = _require(CFG, "location", "clip")

EVENT_YEAR = _require(CFG, "event", "year")
HEATWAVE_WINDOWS = [tuple(w) for w in _require(CFG, "event", "heatwave_windows")]
EXCLUDE_YEARS = CFG.get("event", {}).get("exclude_years", [])

TMAX_THRESH = _require(CFG, "thresholds", "tmax_f")
TMIN_THRESH = _require(CFG, "thresholds", "tmin_f")

WARMING_LEVELS = _require(CFG, "projection", "warming_levels")
WARMING_LEVEL_WINDOW = _require(CFG, "projection", "warming_level_window")
BASELINE_WL = _require(CFG, "projection", "baseline_warming_level")
PERCENTILE = _require(CFG, "projection", "percentile")
WINDOW_YEARS = _require(CFG, "projection", "window_years")
SUBLABELS = CFG.get("projection", {}).get("sublabels") or None

CRAI_ZIP_SHAPEFILE = _require(CFG, "crai", "zip_shapefile")
CRAI_DIR = _require(CFG, "crai", "dir")
CRAI_FILE_SLUG = _require(CFG, "crai", "file_slug")
COUNTY_FIPS = str(_require(CFG, "crai", "county_fips"))
STATE_FIPS = str(CFG.get("crai", {}).get("state_fips", "06"))
GWL_ORDER = _require(CFG, "crai", "gwl_order")
GWL_LABELS = _require(CFG, "crai", "gwl_labels")
GEOID_PREFIX = STATE_FIPS + COUNTY_FIPS

print(f"Loaded config for '{EVENT_NAME}' from {CONFIG_PATH}")

---

## Part 1: Observed Station Data

Pull observed hourly data from the **Historical Data Platform (HDP)** — a
quality-controlled archive of weather station observations — and compute daily max/min
temperature. Station and network come from `location.station_id` / `location.network_id`.

In [ ]:
# Pull all available station data from HDP
cd = ck.ClimateData(verbosity=-1)

station_ds = (
    cd
    .catalog("hdp")
    .network_id(NETWORK_ID)
    .station_id(STATION_ID)
    .get()
)

with ProgressBar():
    station = station_ds.squeeze().compute()

In [ ]:
# Convert temperature K → °F
tas_f = (station["tas"] - 273.15) * 9 / 5 + 32

# Daily aggregates
tas_daily_max_f = tas_f.resample(time="D").max()
tas_daily_min_f = tas_f.resample(time="D").min()

### Event-window statistics

For each configured heat-wave window (`event.heatwave_windows`), compute peak Tmax and
the number of days/nights crossing the configured thresholds. Then rank how rare a
qualifying day (Tmax > `tmax_f` and Tmin >= `tmin_f`) is across the full station record.

In [ ]:
# Isolate the event year
tas_daily_max_event_year = tas_daily_max_f.sel(time=tas_daily_max_f.time.dt.year == EVENT_YEAR)
tas_daily_min_event_year = tas_daily_min_f.sel(time=tas_daily_min_f.time.dt.year == EVENT_YEAR)


def _window_label(i, start, end):
    s = pd.Timestamp(start)
    e = pd.Timestamp(end)
    s_str = s.strftime("%b %-d")
    e_str = e.strftime("%-d") if s.month == e.month else e.strftime("%b %-d")
    return f"Heat Wave {i} ({s_str}–{e_str})"


# Characterize each configured heat-wave window
for i, (start, end) in enumerate(HEATWAVE_WINDOWS, start=1):
    period = slice(start, end)
    label = _window_label(i, start, end)
    hw_tx = tas_daily_max_event_year.sel(time=period)
    hw_tn = tas_daily_min_event_year.sel(time=period)
    days_above = int((hw_tx > TMAX_THRESH).sum())
    peak = float(hw_tx.max())
    nights_above = int((hw_tn >= TMIN_THRESH).sum())
    print(f"{label}")
    print(f"  Peak Tmax                  : {peak:.1f}°F")
    print(f"  Days Tmax > {TMAX_THRESH}°F        : {days_above}")
    print(f"  Nights Tmin >= {TMIN_THRESH}°F     : {nights_above}")
    print()

# Of every day in the full station record, how many had both Tmax > threshold AND Tmin >= threshold?
qualifying = (tas_daily_max_f > TMAX_THRESH) & (tas_daily_min_f >= TMIN_THRESH)
n_qualifying = int(qualifying.sum())
total_days = len(qualifying)

print(f"Days with Tmax > {TMAX_THRESH}°F and Tmin >= {TMIN_THRESH}°F: {n_qualifying} of {total_days} days on record")
pct_rank = 100 - (n_qualifying / total_days * 100)
print(f"A day meeting both thresholds falls in the {pct_rank:.1f}th percentile of all days on record")

### Daily max/min by year

Build a Plotly figure comparing the event year's daily max/min temperatures against
every other year on record, with the configured heat-wave windows and thresholds
annotated.

In [ ]:
# Build daily max temp DataFrame
tmax_df = tas_daily_max_f.to_pandas().reset_index()
tmax_df.columns = ["date", "temp_f"]
tmax_df["year"]      = tmax_df["date"].dt.year
tmax_df["doy"]       = tmax_df["date"].dt.dayofyear
tmax_df["date_str"]  = tmax_df["date"].dt.strftime("%-m/%-d")
tmax_df["series"]    = "Daily Max"

# Build daily min temp DataFrame
tmin_df = tas_daily_min_f.to_pandas().reset_index()
tmin_df.columns = ["date", "temp_f"]
tmin_df["year"]      = tmin_df["date"].dt.year
tmin_df["doy"]       = tmin_df["date"].dt.dayofyear
tmin_df["date_str"]  = tmin_df["date"].dt.strftime("%-m/%-d")
tmin_df["series"]    = "Daily Min"

# Years to make hoverable: first year in record + every year divisible by 5
tmax_hist = tmax_df[tmax_df["year"] != EVENT_YEAR].copy()
tmin_hist = tmin_df[tmin_df["year"] != EVENT_YEAR].copy()
first_yr = tmax_hist["year"].min()
hover_yrs = set(y for y in tmax_hist["year"].unique() if y % 5 == 0) | {first_yr}

In [ ]:
import plotly.graph_objects as go

# ── Leap-safe day-of-year → shared calendar mapping ───────────────────────────
# Anchoring on month/day against a leap year (2000) keeps every year aligned,
# including leap years like the event year itself.
def leap_safe_plot_date(dates):
    return pd.to_datetime({"year": 2000, "month": dates.dt.month, "day": dates.dt.day})


def anchor_2000(date_str):
    """Map an event-window date string onto the shared 2000-anchored calendar."""
    d = pd.Timestamp(date_str)
    return f"2000-{d.month:02d}-{d.day:02d}"


tmax_df   = tmax_df.assign(plot_date=leap_safe_plot_date(tmax_df["date"]))
tmin_df   = tmin_df.assign(plot_date=leap_safe_plot_date(tmin_df["date"]))
tmax_hist = tmax_hist.assign(plot_date=leap_safe_plot_date(tmax_hist["date"]))
tmin_hist = tmin_hist.assign(plot_date=leap_safe_plot_date(tmin_hist["date"]))

lw, alpha = 0.8, 0.3
fig = go.Figure()

# ── Historical background lines — non-hoverable ─────────────────────────────
tmax_hist_bg = tmax_hist[~tmax_hist["year"].isin(hover_yrs)]
for yr, grp in tmax_hist_bg.groupby("year"):
    fig.add_trace(go.Scatter(
        x=grp["plot_date"], y=grp["temp_f"], mode="lines",
        line=dict(color="#e0aaaa", width=lw), opacity=alpha,
        name=str(yr), legendgroup="tmax_hist", showlegend=False,
        hoverinfo="skip",
    ))

tmin_hist_bg = tmin_hist[~tmin_hist["year"].isin(hover_yrs)]
for yr, grp in tmin_hist_bg.groupby("year"):
    fig.add_trace(go.Scatter(
        x=grp["plot_date"], y=grp["temp_f"], mode="lines",
        line=dict(color="#aac4e0", width=lw), opacity=alpha,
        name=str(yr), legendgroup="tmin_hist", showlegend=False,
        hoverinfo="skip",
    ))

# ── Historical background lines — every-5-years, hoverable ─────────────
tmax_hist_hl = tmax_hist[tmax_hist["year"].isin(hover_yrs)]
for yr, grp in tmax_hist_hl.groupby("year"):
    fig.add_trace(go.Scatter(
        x=grp["plot_date"], y=grp["temp_f"], mode="lines",
        line=dict(color="#e0aaaa", width=lw), opacity=alpha,
        name=str(yr), legendgroup="tmax_hist", showlegend=False,
        hovertemplate=f"{yr} Daily Max<br>%{{x|%b %d}}<br>%{{y:.1f}} °F<extra></extra>",
    ))

tmin_hist_hl = tmin_hist[tmin_hist["year"].isin(hover_yrs)]
for yr, grp in tmin_hist_hl.groupby("year"):
    fig.add_trace(go.Scatter(
        x=grp["plot_date"], y=grp["temp_f"], mode="lines",
        line=dict(color="#aac4e0", width=lw), opacity=alpha,
        name=str(yr), legendgroup="tmin_hist", showlegend=False,
        hovertemplate=f"{yr} Daily Min<br>%{{x|%b %d}}<br>%{{y:.1f}} °F<extra></extra>",
    ))

# ── Event year — bold highlighted lines ─────────────────────────────────
tmax_event_df = tmax_df[tmax_df["year"] == EVENT_YEAR]
fig.add_trace(go.Scatter(
    x=tmax_event_df["plot_date"], y=tmax_event_df["temp_f"], mode="lines",
    line=dict(color="firebrick", width=lw * 3), name=f"{EVENT_YEAR} Daily Max",
    hovertemplate=f"{EVENT_YEAR} Daily Max<br>%{{x|%b %d}}<br>%{{y:.1f}} °F<extra></extra>",
))

tmin_event_df = tmin_df[tmin_df["year"] == EVENT_YEAR]
fig.add_trace(go.Scatter(
    x=tmin_event_df["plot_date"], y=tmin_event_df["temp_f"], mode="lines",
    line=dict(color="#1a3f7a", width=lw * 3), name=f"{EVENT_YEAR} Daily Min",
    hovertemplate=f"{EVENT_YEAR} Daily Min<br>%{{x|%b %d}}<br>%{{y:.1f}} °F<extra></extra>",
))

# ── Heat wave bands + labels — one vrect/annotation per configured window ────────
for i, (start, end) in enumerate(HEATWAVE_WINDOWS, start=1):
    x0, x1 = anchor_2000(start), anchor_2000(end)
    fig.add_vrect(x0=x0, x1=x1, fillcolor="darkgoldenrod",
                  opacity=0.12 if i == 1 else 0.20, line_width=0, layer="below")
    fig.add_annotation(
        x=x0, y=35 - (i - 1) * 4, xanchor="center",
        text=_window_label(i, start, end).replace(" (", "<br>("),
        showarrow=False,
        font=dict(size=11, color="darkgoldenrod"), align="center",
    )

# ── Threshold reference lines — labeled directly on the plot instead of the legend ──
fig.add_hline(y=65, line=dict(color="black", width=1.2, dash="dash"),
    annotation_text="65°F AC<br>Threshold", annotation_position="top right",
    annotation_font=dict(size=11, color="black"))
fig.add_hline(y=TMAX_THRESH, line=dict(color="red", width=1.2, dash="dash"),
    annotation_text=f"{TMAX_THRESH}°F Excessive<br>Day Heat", annotation_position="top right",
    annotation_font=dict(size=11, color="red"))
fig.add_hline(y=TMIN_THRESH, line=dict(color="#1a3f7a", width=1.2, dash="dash"),
    annotation_text=f"{TMIN_THRESH}°F Excessive<br>Night Heat", annotation_position="top right",
    annotation_font=dict(size=11, color="#1a3f7a"))

# ── Legend-only entries ────────────────────────────────────────────
# Threshold lines and heat-wave bands are labeled on-plot (annotations above)
# rather than in the legend, to keep the legend short.
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
    line=dict(color="#e0aaaa", width=lw * 2), name="Historical Daily Max (Every 5 Years)"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
    line=dict(color="#aac4e0", width=lw * 2), name="Historical Daily Min (Every 5 Years)"))

# ── Layout ───────────────────────────────────────────────────────
fig.update_layout(
    title=f"{PLACE_NAME} ({STATION_LABEL}) — Daily Max & Min Temperature by Year",
    xaxis=dict(title="", range=["2000-01-01", "2000-12-31"], showgrid=False, fixedrange=True, tickformat="%b"),
    yaxis=dict(title="Temperature (°F)", showgrid=False, fixedrange=True),
    width=815, height=600,
    legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"),
    hovermode="closest",
    dragmode=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=60, r=90, t=50, b=50),
)

fig.show(config=plotly_config)

In [ ]:
fig.write_image("figures/static/fig_daily_temp_by_year_plotly.png", scale=4.0)  # needs kaleido
fig.write_html(
    "figures/html/fig_daily_temp_by_year_plotly.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=plotly_config,
)

---

## Part 2: Historical Context — How Unusual Was This?

Compute the event year's annual mean temperature anomaly relative to the full-record
mean, and compare it against a 5-year rolling mean.

In [ ]:
# Annual mean temperature anomaly — overall baseline
overall_mean = float(tas_f.mean())
tas_daily_mean_f = tas_f.resample(time="D").mean()
annual_mean_f = tas_daily_mean_f.resample(time="YE").mean()
anomaly = annual_mean_f - overall_mean

# Exclude incomplete or unwanted years (event.exclude_years in the config)
if EXCLUDE_YEARS:
    anomaly = anomaly.sel(time=~anomaly.time.dt.year.isin(EXCLUDE_YEARS))

years = anomaly.time.dt.year.values
anom_vals = anomaly.values
rolling_5yr = pd.Series(anom_vals, index=years).rolling(5, center=True).mean()
colors = ["firebrick" if v >= 0 else "steelblue" for v in anom_vals]
idx_event_year = list(years).index(EVENT_YEAR)

In [ ]:
import plotly.graph_objects as go

bar_line_colors = ["gold" if yr == EVENT_YEAR else "rgba(0,0,0,0)" for yr in years]
bar_line_widths = [2.5 if yr == EVENT_YEAR else 0 for yr in years]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=years, y=anom_vals, width=0.8,
    marker=dict(color=colors, opacity=0.75,
                line=dict(color=bar_line_colors, width=bar_line_widths)),
    showlegend=False,
    hovertemplate="%{x}<br>%{y:.2f} °F<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=years, y=rolling_5yr.values, mode="lines",
    line=dict(color="black", width=2), name="5-year rolling mean",
    hovertemplate="5-yr mean<br>%{x}<br>%{y:.2f} °F<extra></extra>",
))

fig.add_hline(y=0, line=dict(color="gray", width=0.8, dash="dash"))

fig.add_annotation(
    x=EVENT_YEAR, y=anom_vals[idx_event_year] + 0.3, text=str(EVENT_YEAR),
    showarrow=False, font=dict(size=12, color="goldenrod"),
)

fig.update_layout(
    title=f"Annual Mean Temperature Anomaly relative to full-record mean at {STATION_LABEL}",
    xaxis=dict(title="Year", showgrid=False, fixedrange=True),
    yaxis=dict(title="Temperature anomaly (°F)", showgrid=False, fixedrange=True),
    width=815, height=600,
    legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"),
    hovermode="closest",
    dragmode=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=60, r=20, t=50, b=50),
)

fig.show(config=plotly_config)

In [ ]:
fig.write_image("figures/static/fig_annual_anomaly_plotly.png", scale=4.0)  # needs kaleido
fig.write_html(
    "figures/html/fig_annual_anomaly_plotly.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=plotly_config,
)

---

## Part 3: How Frequently Will This Occur in the Future?

Using climate model projections from the Cal-Adapt Analytics Engine, estimate how
frequently extreme heat events like this one will occur as the planet warms, across the
warming levels configured in `projection.warming_levels` (typically a recent-past
baseline, present-day, and a future benchmark).

**Methodology:** rather than applying a fixed temperature threshold, extreme heat is
defined relative to the local climate of each simulation:

1. **Threshold calibration at the baseline warming level** (`projection.baseline_warming_level`)
   — for each simulation and each grid cell within the clipped region, compute a high
   percentile (`projection.percentile`) of daily maximum temperature and of daily minimum
   temperature across the calibration window. These thresholds are unique to each
   simulation and grid cell.
2. **Co-occurrence counting** — at each warming level, count days per year where **both**
   daily max temperature exceeds the tmax threshold **and** daily min temperature exceeds
   the tmin threshold simultaneously (hot days with no overnight relief).
3. **Aggregation** — median across all grid cells in the clipped region, then median
   across model simulations, to produce a robust regional estimate at each warming level.

Uses dynamically-downscaled output from the **WRF models** (UCLA, 3 km resolution,
SSP3-7.0 scenario).

### Pulling WRF tasmax and tasmin

The clip region (`location.clip`) and warming levels (`projection.warming_levels`) are
set in the config.

In [ ]:
# Daily max temperature for the clipped region at the configured warming levels
cd.reset()
tasmax_ds = (
    cd
    .catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d03")
    .variable("t2max")
    .processes({
        "warming_level": {
            "warming_levels": WARMING_LEVELS,
            "warming_level_window": WARMING_LEVEL_WINDOW,
        },
        "clip": CLIP,
        "convert_units": "degF",
    })
    .get()
)

# Daily min temperature — same query, different variable
cd.reset()
tasmin_ds = (
    cd
    .catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d03")
    .variable("t2min")
    .processes({
        "warming_level": {
            "warming_levels": WARMING_LEVELS,
            "warming_level_window": WARMING_LEVEL_WINDOW,
        },
        "clip": CLIP,
        "convert_units": "degF",
    })
    .get()
)

In [ ]:
with ProgressBar():
    t2max = tasmax_ds["t2max"].compute()
    t2min = tasmin_ds["t2min"].compute()

# ---------------------------------------------------------------------------
# Thresholds: PERCENTILE calibrated at the baseline warming level.
# Reduces over time_delta only → shape (sim, y, x).
# Broadcasting against t2max/t2min [(sim, warming_level, time_delta, y, x)]
# applies the same per-sim, per-cell threshold across all warming levels.
# ---------------------------------------------------------------------------
tmax_threshold = (
    t2max.sel(warming_level=BASELINE_WL)
    .quantile(PERCENTILE, dim="time_delta")
    .drop_vars("quantile")
)
tmin_threshold = (
    t2min.sel(warming_level=BASELINE_WL)
    .quantile(PERCENTILE, dim="time_delta")
    .drop_vars("quantile")
)

print(f"tmax threshold shape: {tmax_threshold.shape}  (sim, y, x)")
print(f"tmin threshold shape: {tmin_threshold.shape}")

In [ ]:
# A day qualifies only when BOTH tmax and tmin exceed their respective thresholds
# (extreme heat with no overnight relief). xarray broadcasts (sim, y, x) thresholds
# across the full (sim, warming_level, time_delta, y, x) arrays automatically.
events = (t2max > tmax_threshold) & (t2min > tmin_threshold)

In [ ]:
# Result: one events-per-year value per warming level
events_per_year = (
    events
    .sum(dim="time_delta")
    .mean(dim=["x", "y"])
    .mean(dim="sim")
    / WINDOW_YEARS
)

warming_levels = events_per_year.warming_level.values
rates = events_per_year.values
labels = [f"{wl}°C" for wl in warming_levels]
colors = ["#4393c3", "#f4a582", "#d6604d"][:len(labels)]  # cool → warm palette

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Bar(
    x=labels, y=rates,
    marker=dict(color=colors, line=dict(color="white", width=0.8)),
    text=[f"{r:.2f} days/yr" for r in rates],
    textposition="outside",
    textfont=dict(size=11),
    showlegend=False,
    hovertemplate="%{x}<br>%{y:.2f} days/yr<extra></extra>",
))

fig.update_layout(
    title="Extreme Heat Days Without Nighttime Reprieve",
    xaxis=dict(title="Global Warming Level (°C)", showgrid=False, fixedrange=True),
    yaxis=dict(title="Extreme heat days per year", showgrid=False, fixedrange=True,
               range=[0, max(rates) * 1.35]),
    width=700, height=500,
    hovermode="closest",
    dragmode=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=60, r=20, t=70, b=50),
)

fig.show(config=plotly_config)

In [ ]:
fig.write_image("figures/static/fig_events_by_gwl_plotly.png", scale=4.0)  # needs kaleido
fig.write_html(
    "figures/html/fig_events_by_gwl_plotly.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=plotly_config,
)

In [ ]:
# Convert days/year → return period (1-in-X-years)
# A rate of 0.5 days/yr means this type of day happens once every 2 years on average.
return_periods = 1 / rates

sublabels = SUBLABELS if SUBLABELS and len(SUBLABELS) == len(labels) else labels

In [ ]:
import plotly.graph_objects as go

tick_labels_plotly = [f"{s}<br>({l})" for l, s in zip(labels, sublabels)]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=tick_labels_plotly, y=return_periods,
    marker=dict(color=colors, line=dict(color="white", width=0.8)),
    text=[f"1-in-{rp:.0f} yrs" for rp in return_periods],
    textposition="outside",
    textfont=dict(size=11),
    showlegend=False,
    hoverinfo="skip",
))

fig.update_layout(
    title="How Much More Frequent Will Days Like This Be?",
    xaxis=dict(title="Global Warming Level (°C)", showgrid=False, fixedrange=True),
    yaxis=dict(title="Return period (years)", showgrid=False, fixedrange=True,
               range=[0, max(return_periods) * 1.25]),
    width=700, height=450,
    hovermode="closest",
    dragmode=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=60, r=20, t=50, b=50),
)

fig.show(config=plotly_config)

In [ ]:
fig.write_image("figures/static/fig_return_period_by_gwl_plotly.png", scale=4.0)  # needs kaleido
fig.write_html(
    "figures/html/fig_return_period_by_gwl_plotly.html",
    include_plotlyjs="cdn",
    full_html=False,
    config=plotly_config,
)

---

## Part 4: CRAI Layers — Tract-Level Choropleth Maps

Reads pre-generated CRAI hazard CSVs for the region (see `crai` in the config) — they
are **not** regenerated by this notebook. You must supply matching files for a new
location before running this section; see `crai.dir` / `crai.file_slug` in
`event_config.yaml`.

Builds two raw-count choropleths (day-heat/night-heat co-occurrence, and consecutive
heatwave count) across the configured warming levels, then two delta maps showing the
change in each metric relative to the baseline warming level.

In [ ]:
import os
import json
import pandas as pd
import geopandas as gpd
from shapely import wkt
import plotly.graph_objects as go

plotly_config = dict(
    displaylogo=False,
    scrollZoom=False,
    modeBarButtonsToRemove=[
        "zoom2d", "pan2d", "select2d", "lasso2d",
        "zoomIn2d", "zoomOut2d", "autoScale2d", "resetScale2d",
        "zoomInGeo", "zoomOutGeo", "resetGeo", "hoverClosestGeo",
    ],
)

# ── Standalone setup — this cell doesn't depend on any cell above it ──
zip_codes = gpd.read_file(CRAI_ZIP_SHAPEFILE).to_crs('EPSG:4326')

crai_path = CRAI_DIR
raw_counts_path = os.path.join(crai_path, 'raw_counts')
geometry_source = os.path.join(crai_path, 'RECENT_HAZARDS_crai_heat_counts_gwl2.csv')

gwl_order = GWL_ORDER
gwl_labels = GWL_LABELS

# Load the county's tract geometry once (raw_counts files have no geometry column).
# COUNTYFP is read as a string throughout this section to avoid the int/str
# mismatch that this template's original single-city version had.
geom_chunks = []
with pd.read_csv(
    geometry_source,
    usecols=['GEOID', 'COUNTYFP', 'geometry'],
    dtype={'GEOID': str, 'COUNTYFP': str},
    chunksize=100000,
) as reader:
    for chunk in reader:
        geom_chunks.append(chunk[chunk['COUNTYFP'] == COUNTY_FIPS])

county_geom = pd.concat(geom_chunks, ignore_index=True)
county_geom['geometry'] = county_geom['geometry'].apply(wkt.loads)
county_geom = gpd.GeoDataFrame(county_geom[['GEOID', 'geometry']], geometry='geometry', crs='EPSG:4326')

# Map each census tract to the zip code its centroid falls within, so hover
# labels can show a recognizable zip code instead of the raw GEOID. Built once
# here and reused (by GEOID) regardless of which data source (raw_counts vs.
# CRAI hazard-score files) a given trace's geometry actually came from.
_tract_centroids = county_geom.copy()
_utm_crs = county_geom.estimate_utm_crs()
_tract_centroids['geometry'] = (
    _tract_centroids.geometry.to_crs(_utm_crs).centroid.to_crs('EPSG:4326')
)
_tract_zip = gpd.sjoin(
    _tract_centroids, zip_codes[['ZIP5', 'PO_NAME', 'geometry']],
    how='left', predicate='within',
)
_tract_zip['zip_label'] = (
    _tract_zip['ZIP5'].fillna('Unknown')
    + ' (' + _tract_zip['PO_NAME'].fillna('Unknown').str.title() + ')'
)
geoid_to_zip_label = dict(zip(_tract_zip['GEOID'], _tract_zip['zip_label']))


def load_raw_counts(metric_prefix, value_col):
    """Load a raw_counts metric across all GWLs, restricted to the configured county."""
    layers = {}
    for gwl in gwl_order:
        df = pd.read_csv(
            os.path.join(raw_counts_path, f'{metric_prefix}_{gwl}.csv'),
            dtype={'GEOID': str},
        )
        df['GEOID'] = df['GEOID'].str.zfill(11)
        df = df[df['GEOID'].str.startswith(GEOID_PREFIX)]
        layers[gwl] = county_geom.merge(df[['GEOID', value_col]], on='GEOID', how='inner')
    return layers


tx105_layers = load_raw_counts(f'climate_TX105day_75night_{CRAI_FILE_SLUG}', 'heat_counts')

# ── Plotly choropleth-with-tabs ──────────────────────────────────────────────
# Simplify before embedding: each Choropleth trace embeds its own full copy of
# the geojson (Plotly doesn't dedupe shared geojson across traces), so at
# full precision this blows up to ~5MB+ of redundant coordinates across the
# GWL traces. A ~5m tolerance is invisible at web-map scale but cuts geojson
# size ~4x, and does the same for the zip-boundary point count.
_SIMPLIFY_TOL = 0.00005  # degrees, ≈5m

_county_geom_simplified = county_geom.copy()
_county_geom_simplified["geometry"] = _county_geom_simplified.geometry.simplify(
    _SIMPLIFY_TOL, preserve_topology=True
)
county_geojson = json.loads(_county_geom_simplified.to_json())


def _zip_boundary_trace(linewidth=1.5):
    zc_simplified = zip_codes.geometry.simplify(_SIMPLIFY_TOL, preserve_topology=True)
    lons, lats = [], []
    for geom in zc_simplified.boundary:
        parts = geom.geoms if geom.geom_type == "MultiLineString" else [geom]
        for part in parts:
            x, y = part.xy
            lons.extend([round(v, 5) for v in x] + [None])
            lats.extend([round(v, 5) for v in y] + [None])
    return go.Scattergeo(
        lon=lons, lat=lats, mode="lines",
        line=dict(color="#404040", width=linewidth),
        hoverinfo="skip", showlegend=False,
    )


def save_tabbed_choropleth_plotly(layers, value_col, title, html_path, png_path, hover_label=None):
    """One Choropleth trace per GWL, toggled via native Plotly buttons (real tabs,
    restyling trace visibility), with a zip-code boundary overlay on the same geo
    subplot. `png_path` is explicit per call so different metrics never clobber
    each other's static export."""
    vmax = max(layers[gwl][value_col].max() for gwl in gwl_order)

    fig = go.Figure()

    for i, gwl in enumerate(gwl_order):
        gdf = layers[gwl]
        fig.add_trace(go.Choropleth(
            geojson=county_geojson, featureidkey="properties.GEOID",
            locations=gdf["GEOID"], z=gdf[value_col],
            zmin=0, zmax=vmax, colorscale="YlOrRd",
            marker=dict(line=dict(color="#cccccc", width=0.4)),
            colorbar=dict(
                title=dict(text="# of Days", font=dict(size=16)),
                tickfont=dict(size=16),
                thickness=20, len=0.7, x=1.0, xanchor="left",
            ),
            visible=(i == 0),
            name=gwl_labels[gwl],
            customdata=gdf["GEOID"].map(geoid_to_zip_label),
            hovertemplate="%{customdata}<br>" + (hover_label or value_col) + ": %{z:.2f}<extra></extra>",
        ))

    fig.add_trace(_zip_boundary_trace())

    n = len(gwl_order)
    buttons = []
    for i, gwl in enumerate(gwl_order):
        visible = [j == i for j in range(n)] + [True]  # zip-boundary trace always on
        buttons.append(dict(
            label=gwl_labels[gwl],
            method="update",
            args=[{"visible": visible}],
        ))

    # `fitbounds="locations"` pads the view a lot more than it needs to, which
    # is why the map reads as "zoomed out" — compute the actual tract bounding
    # box instead and set the geo ranges directly, with just a small buffer.
    minx, miny, maxx, maxy = county_geom.total_bounds
    lon_pad = (maxx - minx) * 0.03
    lat_pad = (maxy - miny) * 0.03

    fig.update_geos(
        visible=False, bgcolor="white",
        projection_type="mercator",
        lonaxis_range=[minx - lon_pad, maxx + lon_pad],
        lataxis_range=[miny - lat_pad, maxy + lat_pad],
    )

    fig.update_layout(
        title=dict(
            text=title,
            x=0.043, xanchor="left", y=0.99, yanchor="top", font=dict(size=20),
            pad=dict(t=20)
        ),
        updatemenus=[dict(
            type="buttons", direction="right",
            buttons=buttons,
            # Sits just above the geo domain's top edge — right above the map,
            # not overlapping it.
            x=0.02, xanchor="left", y=1, yanchor="bottom",
            showactive=True,
            bgcolor="#f8f9fa", bordercolor="#999",
            font=dict(size=14),
            pad=dict(t=8, b=8, l=10, r=10),
        )],
        width=815, height=810,
        paper_bgcolor="white",
        margin=dict(l=10, r=80, t=100, b=10),
        hovermode="closest",
    )

    fig.write_html(html_path, include_plotlyjs="cdn", full_html=False, config=plotly_config)
    fig.write_image(png_path, scale=4.0)  # needs kaleido
    fig.show(config=plotly_config)
    return fig


# Day-heat / night-heat co-occurrence metric — native Plotly tabs toggle between warming levels
tx105_fig_plotly = save_tabbed_choropleth_plotly(
    tx105_layers, "heat_counts", f"{TMAX_THRESH}°F Day / {TMIN_THRESH}°F Night",
    "figures/html/fig_tx105_75night_by_gwl_plotly.html",
    "figures/static/fig_tx105_75night_by_gwl_plotly.png",
    hover_label="Extreme Days",
)

In [ ]:
# Consecutive heatwave metric — native Plotly tabs toggle between warming levels.
# Reuses load_raw_counts / gwl_order / gwl_labels / save_tabbed_choropleth_plotly
# from the cell above (run that cell first).
heatwave_layers = load_raw_counts(f'climate_consecutive_heatwave_{CRAI_FILE_SLUG}', 'heatwave_days')
heatwave_fig_plotly = save_tabbed_choropleth_plotly(
    heatwave_layers, 'heatwave_days', 'Consecutive Heatwave Count',
    'figures/html/fig_consecutive_heatwave_by_gwl_plotly.html',
    'figures/static/fig_consecutive_heatwave_by_gwl_plotly.png',
)

### Delta maps — change vs. baseline warming level

Same tract-level maps as above, but showing the change in each metric relative to the
baseline warming level's CRAI hazard score, sourced from the pre-generated
`RECENT_HAZARDS_crai_*` files in `crai.dir`.

In [ ]:
import numpy as np

# Delta versions — change vs. the baseline GWL, sourced from the CRAI hazard-score
# files (RECENT_HAZARDS_crai_heat_counts_gwl*.csv / RECENT_HAZARDS_crai_heatwave_days_gwl*.csv
# in crai_path), NOT the raw_counts/ folder. Reuses crai_path / gwl_order / gwl_labels /
# zip_codes / _zip_boundary_trace / _SIMPLIFY_TOL from the cells above (run those first).

metric_config = {
    'heat_counts': 'RECENT_HAZARDS_crai_heat_counts_',
    'heatwave_days': 'RECENT_HAZARDS_crai_heatwave_days_',
}


def load_hazard_csv(filepath, filename, chunk_size=100000, crs='EPSG:4326'):
    full_path = os.path.join(filepath, filename)
    processed_chunks = []
    with pd.read_csv(
        full_path, chunksize=chunk_size, dtype={'GEOID': str, 'COUNTYFP': str},
    ) as reader:
        for chunk in reader:
            processed_chunks.append(chunk)

    df = pd.concat(processed_chunks, ignore_index=True)
    # Force zero-padded 11-digit GEOID (matches the raw_counts loader's
    # convention) so it lines up with geoid_to_zip_label for hover labels.
    df['GEOID'] = df['GEOID'].str.zfill(11)

    def safe_wkt(x):
        try:
            return wkt.loads(x)
        except Exception:
            return None

    df['geometry'] = df['geometry'].apply(safe_wkt)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=crs)
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[gdf['COUNTYFP'] == COUNTY_FIPS]
    return gdf


def save_delta_tabbed_choropleth_plotly(metric, title, html_path, col_name='heat_hazard_score'):
    """One Choropleth trace per non-baseline GWL showing its delta vs. the baseline
    GWL, toggled via native Plotly buttons (real tabs), with the same zip-code
    boundary overlay."""
    prefix = metric_config[metric]
    layers = {
        gwl: load_hazard_csv(crai_path, f'{prefix}{gwl}.csv')
        for gwl in gwl_order
    }

    baseline_key = gwl_order[0]
    compare_keys = [gwl for gwl in gwl_order if gwl != baseline_key]

    base = layers[baseline_key][['GEOID', col_name, 'geometry']]
    deltas = {}
    for compare_key in compare_keys:
        compare = layers[compare_key][['GEOID', col_name]]
        merged = base.merge(compare, on='GEOID', suffixes=(f'_{baseline_key}', f'_{compare_key}'))
        merged['delta'] = merged[f'{col_name}_{compare_key}'] - merged[f'{col_name}_{baseline_key}']
        deltas[compare_key] = gpd.GeoDataFrame(merged, geometry='geometry', crs=layers[baseline_key].crs)

    vmax = max(np.nanmax(np.abs(deltas[k]['delta'])) for k in compare_keys)

    # Same baseline geometry underlies every tab — simplify and embed once.
    geom_simplified = deltas[compare_keys[0]][['GEOID', 'geometry']].copy()
    geom_simplified['geometry'] = geom_simplified.geometry.simplify(_SIMPLIFY_TOL, preserve_topology=True)
    delta_geojson = json.loads(geom_simplified.to_json())

    fig = go.Figure()

    for i, compare_key in enumerate(compare_keys):
        gdf = deltas[compare_key]
        fig.add_trace(go.Choropleth(
            geojson=delta_geojson, featureidkey="properties.GEOID",
            locations=gdf["GEOID"], z=gdf["delta"],
            zmin=-vmax, zmax=vmax, colorscale="RdBu", reversescale=True,
            marker=dict(line=dict(color="#cccccc", width=0.4)),
            colorbar=dict(
                tickmode="array",
                tickvals=[-0.75 * vmax, 0.75 * vmax],
                ticktext=["Lower <br>Risk", "Higher <br>Risk"],
                tickfont=dict(size=14),
                thickness=20, len=0.7, x=1.0, xanchor="left",
            ),
            visible=(i == 0),
            name=gwl_labels[compare_key],
            customdata=gdf["GEOID"].map(geoid_to_zip_label),
            hovertemplate="%{customdata}<br>Δ: %{z:.2f}<extra></extra>",
        ))

    fig.add_trace(_zip_boundary_trace())

    n = len(compare_keys)
    buttons = []
    for i, compare_key in enumerate(compare_keys):
        visible = [j == i for j in range(n)] + [True]
        buttons.append(dict(
            label=gwl_labels[compare_key],
            method="update",
            args=[{"visible": visible}],
        ))

    # Same tight-bounds + mercator treatment as save_tabbed_choropleth_plotly —
    # `fitbounds="locations"` pads more than it needs to and equirectangular
    # (the default) stretches shapes east-west at this latitude.
    minx, miny, maxx, maxy = deltas[compare_keys[0]].total_bounds
    lon_pad = (maxx - minx) * 0.03
    lat_pad = (maxy - miny) * 0.03

    fig.update_geos(
        visible=False, bgcolor="white",
        projection_type="mercator",
        lonaxis_range=[minx - lon_pad, maxx + lon_pad],
        lataxis_range=[miny - lat_pad, maxy + lat_pad],
    )

    fig.update_layout(
        title=dict(
            text=title,
            x=0.043, xanchor="left", y=0.99, yanchor="top", font=dict(size=20),
            pad=dict(t=20),
        ),
        updatemenus=[dict(
            type="buttons", direction="right",
            buttons=buttons,
            x=0.02, xanchor="left", y=1, yanchor="bottom",
            showactive=True,
            bgcolor="#f8f9fa", bordercolor="#999",
            font=dict(size=14),
            pad=dict(t=8, b=8, l=10, r=10),
        )],
        width=815, height=810,
        paper_bgcolor="white",
        margin=dict(l=10, r=80, t=100, b=10),
    )

    fig.write_html(html_path, include_plotlyjs="cdn", full_html=False, config=plotly_config)
    fig.show(config=plotly_config)
    return fig


# Day-heat / night-heat co-occurrence metric — change in heat hazard score vs baseline GWL
tx105_delta_fig_plotly = save_delta_tabbed_choropleth_plotly(
    'heat_counts', 'Hot Days and Warm Nights - Difference from Baseline',
    'figures/html/fig_tx105_75night_delta_by_gwl_plotly.html',
)

In [ ]:
# Consecutive heatwave metric — delta vs. baseline GWL.
# Reuses save_delta_tabbed_choropleth_plotly from the cell above (run that cell first).
heatwave_delta_fig_plotly = save_delta_tabbed_choropleth_plotly(
    'heatwave_days', 'Consecutive Heatwaves - Difference from Baseline',
    'figures/html/fig_consecutive_heatwave_delta_by_gwl_plotly.html',
)